In [69]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from utilities.clean import report, find_outliers

df = pd.read_csv("./data/train.csv")
print(f"Loaded {len(df):,} rows")

%load_ext autoreload
%autoreload 2

Loaded 891 rows
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [70]:
print('Before Cleaning:')
report(df)

Before Cleaning:
Total rows: 891, Total columns: 12

Duplicate rows: 0

Missing values:
Age: 177 missing values
Cabin: 687 missing values
Embarked: 2 missing values

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB


In [71]:
df = df[['Survived', 'Pclass',  'Sex', 'Age', 'Fare']].copy()
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
print(df.dtypes)

Survived      int64
Pclass        int64
Sex           int64
Age         float64
Fare        float64
dtype: object


In [72]:
find_outliers(df, 'Age')
median_age = df['Age'].median()
df['Age'] = df['Age'].fillna(median_age)
df['Age'] = df['Age'].astype(int)
print(df.dtypes)

Mean: 29.69911764705882
Median: 28.0
Outliers found: 11
Survived      int64
Pclass        int64
Sex           int64
Age           int64
Fare        float64
dtype: object


In [73]:
print('After Cleaning:')
report(df)

After Cleaning:
Total rows: 891, Total columns: 5

Duplicate rows: 133

No missing values

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    int64  
 3   Age       891 non-null    int64  
 4   Fare      891 non-null    float64
dtypes: float64(1), int64(4)
memory usage: 34.9 KB


In [74]:
X = df[['Pclass', 'Sex', 'Age', 'Fare']]
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [75]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)
print(f"Train Accuracy: {train_acc:.3f}")
print(f"Test Accuracy: {test_acc:.3f}")

Train Accuracy: 0.791
Test Accuracy: 0.804


In [76]:
# Checking how it did
from sklearn.metrics import accuracy_score, confusion_matrix

preds = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, preds))
print(confusion_matrix(y_test, preds))

Accuracy: 0.8044692737430168
[[90 15]
 [20 54]]


In [80]:
# Interpret the coefficients
coefficients = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': model.coef_[0],
    'odds_ratio': np.exp(model.coef_[0]),
})

coefficients['abs_coef'] = coefficients['coefficient'].abs()
coefficients = coefficients.sort_values(by='abs_coef', ascending=False).drop(columns='abs_coef')

print(coefficients.to_string(index=False))

feature  coefficient  odds_ratio
    Sex     2.465870   11.773718
 Pclass    -0.996629    0.369122
    Age    -0.025195    0.975119
   Fare     0.001181    1.001182


## Final check on test data

In [84]:
test_df = pd.read_csv("./data/test.csv")
X_final = df[['Pclass', 'Sex', 'Age', 'Fare']]

final_predictions = model.predict(X_final)